# Random Forest Model for Theft Rate Prediction

This notebook builds a Random Forest classifier to predict theft levels in a supermarket dataset based on features such as foot traffic, staff count, hour of day, and high-value items.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 2. Load and Explore the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('/Users/dhanavijayan/UI/supermarket_sample.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
print(df.describe())

In [ ]:
# Visualize the distribution of Theft_Level (target variable)
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
df['Theft_Level'].value_counts().plot(kind='bar', color=['green', 'orange', 'red'])
plt.title('Distribution of Theft Level')
plt.xlabel('Theft Level')
plt.ylabel('Count')
plt.xticks(rotation=0)

plt.subplot(1, 2, 2)
df['Theft_Level'].value_counts(normalize=True).plot(kind='pie', autopct='%1.1f%%', colors=['green', 'orange', 'red'])
plt.title('Proportion of Theft Levels')
plt.ylabel('')

plt.tight_layout()
plt.show()

print("\nTheft Level Counts:")
print(df['Theft_Level'].value_counts())

## 3. Data Preprocessing and Feature Engineering

In [ ]:
# Prepare data for modeling
# Select features (exclude Row_ID and columns that are outputs/residuals)
X = df[['Foot_Traffic', 'Staff_Count', 'Hour', 'High_Value_Items', 'Shrinkage_INR']]
y = df['Theft_Level']

print("Features used for modeling:")
print(X.columns.tolist())
print("\nFeature data types:")
print(X.dtypes)

# Check for missing values
print("\nMissing values in features:")
print(X.isnull().sum())
print("\nMissing values in target:")
print(y.isnull().sum())

# Encode the target variable (Theft_Level)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("\nLabel Encoding Mapping:")
for i, class_name in enumerate(le.classes_):
    print(f"  {class_name}: {i}")

print("\nEncoded target variable unique values:", np.unique(y_encoded))

## 4. Split Data into Training and Testing Sets

In [ ]:
# Split the data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")
print(f"\nTraining set class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u}: {c} samples ({c/len(y_train)*100:.1f}%)")
print(f"\nTesting set class distribution:")
unique, counts = np.unique(y_test, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u}: {c} samples ({c/len(y_test)*100:.1f}%)")

## 5. Train the Random Forest Model

In [ ]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,           # Number of trees in the forest
    max_depth=15,               # Maximum depth of each tree
    min_samples_split=5,        # Minimum samples required to split
    min_samples_leaf=2,         # Minimum samples in leaf node
    random_state=42,
    n_jobs=-1                   # Use all available processors
)

print("Training the Random Forest model...")
rf_model.fit(X_train, y_train)
print("Model training completed!")

print(f"\nModel Parameters:")
print(f"  Number of estimators: {rf_model.n_estimators}")
print(f"  Max depth: {rf_model.max_depth}")
print(f"  Number of features: {rf_model.n_features_in_}")
print(f"  Number of classes: {rf_model.n_classes_}")
print(f"  Classes: {rf_model.classes_}")

## 6. Evaluate Model Performance

In [ ]:
# Make predictions on training and testing sets
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

# Calculate performance metrics on training set
train_accuracy = accuracy_score(y_train, y_train_pred)
print("=" * 50)
print("TRAINING SET PERFORMANCE")
print("=" * 50)
print(f"Accuracy: {train_accuracy:.4f}")
print(f"\nDetailed Classification Report:")
print(classification_report(y_train, y_train_pred, target_names=le.classes_))

# Calculate performance metrics on testing set
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, average='weighted')
test_recall = recall_score(y_test, y_test_pred, average='weighted')
test_f1 = f1_score(y_test, y_test_pred, average='weighted')

print("=" * 50)
print("TESTING SET PERFORMANCE")
print("=" * 50)
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-Score:  {test_f1:.4f}")
print(f"\nDetailed Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=le.classes_))

In [ ]:
# Visualize Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print("Confusion Matrix:")
print(cm)

In [ ]:
# Visualize Feature Importance
feature_importance = rf_model.feature_importances_
feature_names = X.columns

# Sort features by importance
sorted_idx = np.argsort(feature_importance)[::-1]

plt.figure(figsize=(10, 6))
plt.title('Feature Importance in Random Forest Model')
plt.bar(range(len(sorted_idx)), feature_importance[sorted_idx], align='center')
plt.xticks(range(len(sorted_idx)), [feature_names[i] for i in sorted_idx], rotation=45, ha='right')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

print("Feature Importance Ranking:")
for i, idx in enumerate(sorted_idx, 1):
    print(f"{i}. {feature_names[idx]}: {feature_importance[idx]:.4f}")

## 7. Make Predictions on New Data

In [ ]:
# Example: Make predictions on new data
# Create sample new data
new_data = pd.DataFrame({
    'Foot_Traffic': [300, 150, 450, 100],
    'Staff_Count': [4, 6, 8, 3],
    'Hour': [15, 10, 18, 9],
    'High_Value_Items': [50, 75, 90, 20],
    'Shrinkage_INR': [500, 300, 800, 200]
})

print("New Data for Prediction:")
print(new_data)
print("\n" + "=" * 50)

# Get predictions
new_predictions_encoded = rf_model.predict(new_data)
new_predictions = le.inverse_transform(new_predictions_encoded)

# Get prediction probabilities
prediction_probs = rf_model.predict_proba(new_data)

# Create results dataframe
results = pd.DataFrame({
    'Foot_Traffic': new_data['Foot_Traffic'],
    'Staff_Count': new_data['Staff_Count'],
    'Hour': new_data['Hour'],
    'High_Value_Items': new_data['High_Value_Items'],
    'Predicted_Theft_Level': new_predictions
})

print("\nPrediction Results:")
print(results)

print("\n" + "=" * 50)
print("Prediction Probabilities:")
prob_df = pd.DataFrame(
    prediction_probs,
    columns=[f'{class_name}_prob' for class_name in le.classes_]
)
print(prob_df)

print("\n" + "=" * 50)
print("\nInterpretation:")
for i, (idx, row) in enumerate(results.iterrows()):
    print(f"\nSample {i+1}:")
    print(f"  Foot Traffic: {row['Foot_Traffic']}, Staff: {row['Staff_Count']}, Hour: {row['Hour']}")
    print(f"  Predicted Theft Level: {row['Predicted_Theft_Level']}")
    print(f"  Confidence: {prob_df.loc[i].max():.2%}")

## 8. Save the Trained Model as PKL

In [ ]:
# Save the trained model and label encoder as a PKL artifact
MODEL_DIR = Path('artifacts')
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / 'theft_rate_model.pkl'

artifact = {
    'model': rf_model,
    'label_encoder': le,
    'feature_columns': X.columns.tolist(),
}

joblib.dump(artifact, MODEL_PATH)
print(f'Model saved to: {MODEL_PATH.resolve()}')